# Практикум 3. Разработка и оценка моделей для финансовых рынков

**Курс «Анализ данных и машинное обучение на финансовых рынках»**

---

### Как работать с этим блокнотом

1. Выполняйте ячейки **по порядку**: `Shift + Enter`.
2. Подойдёт Jupyter (Anaconda или python.org + pip) или Google Colab — см. инструкцию в информационном
   разделе курса. Нужны numpy, pandas, scipy, matplotlib; sklearn, statsmodels и torch — по желанию (ячейки
   с ними необязательны и пропускаются, если библиотеки нет).
3. 📘 — разбор: запустите и посмотрите результат. ✏️ — задание: допишите код вместо `...` или `# ваш код`, затем запустите
   ячейку проверки с `assert` — если ошибок нет, решение верно.
   💡 — решение под треугольником, открывайте после честной попытки.
4. Перед сдачей: **Kernel → Restart & Run All**.

### Что будет к концу блокнота

Вы проверите модель по времени (walk-forward), увидите, как случайная кросс-валидация и утечки
завышают качество, посчитаете метрики, ROC и доверительный интервал AUC, откалибруете вероятности, подберёте
гиперпараметры, объясните модель перестановочной важностью, проведёте бэктест с издержками и настроите мониторинг
и правило остановки. Каждое задание проверяется ячейкой с `assert`.

In [ ]:
import json, os, sys, time, random
from datetime import datetime, timedelta, timezone
from pprint import pprint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 140)
BLUE, ORANGE = "#34519E", "#CF7A2A"      # цвета серий на графиках
NAVY, GREY = "#1E2761", "#9AA3B5"
IN_COLAB = "google.colab" in sys.modules
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| Colab" if IN_COLAB else "| локально")

Данные — условная фондовая площадка **«Меридиан»** (все компании вымышлены), затем — ячейка **«Инструменты курса»** (функции тем 1–2: `make_features`, `fit_logit`, `auc`, `roc_points`, `fit_forest`, `walk_forward`, `sharpe`, `max_drawdown` и др.).

In [ ]:
# Сквозной кейс курса «Анализ данных и машинное обучение на финансовых рынках» — «Меридиан»: условная
# фондовая площадка. Все компании, тикеры, котировки и новости вымышлены, совпадения с реальными компаниями
# случайны; генерация детерминирована (seed 2026), у всех получаются одинаковые числа. Только стандартная библиотека.
#   STOCKS      — 12 акций четырёх секторов: тикер, название, сектор, число акций (млн), лот
#   DATES       — торговые дни 09.01.2019–30.09.2026 (без выходных и праздников)
#   PRICES      — дневные данные акций: date, ticker, open, high, low, close (без корректировок), adj_close
#                 (с учётом дивидендов и сплитов), volume (шт.)
#   INDEX       — индекс широкого рынка MRDX: date, open, high, low, close
#   FX          — условный курс доллара: date, usd
#   NEWS        — новости компаний (кроме KLSU и TVRA): date, ticker, headline, label (разметка аналитика: позитив/негатив/нейтрально)
#   DIVIDENDS, SPLITS — корпоративные действия (дата отсечки, размер дивиденда на акцию; коэффициент сплита)
#   RAW_PRICES  — «сырая» выгрузка котировок трёх акций с типичными дефектами (ключ — RAW_DEFECTS)
#   ticks(ticker, start, end) — ЛЕНИВЫЙ генератор сделок внутри дня (время, цена, количество, сторона)
import math as _m
import random as _random
from datetime import date as _date, timedelta as _td

_SEED = 2026
_TITLE = "Меридиан"
_INDEX = "MRDX"
_R = _random.Random(_SEED)
_D0, _D1 = _date(2019, 1, 9), _date(2026, 9, 30)

_STOCKS = [  # тикер, название, сектор, бета рынка, бета курса, σ собств., лаг рынка, цена, акций (млн), див. доходн.
    ("BRZB", "Бриз-Банк", "финансы", 1.20, -0.10, 0.012, 0.00, 250, 2400, 0.08),
    ("AQLF", "Аквилон Финанс", "финансы", 1.05, -0.05, 0.014, 0.00, 85, 3100, 0.06),
    ("PKRV", "Покров Страхование", "финансы", 0.85, 0.00, 0.011, 0.00, 410, 420, 0.07),
    ("GRZN", "Горизонт-Нефть", "энергетика", 0.95, 0.55, 0.011, 0.00, 640, 1500, 0.10),
    ("SRMG", "Сарматгаз", "энергетика", 0.85, 0.45, 0.013, 0.00, 175, 5200, 0.09),
    ("VLTE", "Вольта Энерго", "энергетика", 0.70, 0.20, 0.010, 0.00, 32, 22000, 0.07),
    ("KLSU", "Колос-Юг", "агро", 0.60, 0.15, 0.015, 0.35, 120, 900, 0.04),
    ("TVRA", "Таврида-Агро", "агро", 0.60, 0.15, 0.015, 0.35, 118, 760, 0.00),
    ("NVGR", "Ниваград", "агро", 0.55, 0.10, 0.016, 0.35, 56, 1300, 0.03),
    ("PXLS", "Пиксель Софт", "технологии", 1.35, -0.30, 0.017, 0.00, 2400, 120, 0.00),
    ("NSFT", "Ноосфера Телеком", "технологии", 1.10, -0.20, 0.013, 0.00, 310, 1700, 0.02),
    ("KBRM", "Кибериум", "технологии", 1.45, -0.35, 0.020, 0.00, 95, 2600, 0.00)]
_LAG_UNTIL = _date(2024, 6, 30)       # после этой даты запаздывание агроакций почти исчезает (дрейф)
_LAG_AFTER = 0.0
_SPLIT = ("PXLS", _date(2023, 3, 15), 10)
_STRESS = {_date(2020, 3, 9): -4.0, _date(2020, 3, 12): -3.0, _date(2020, 3, 16): -3.5}
_HOLI = {(1, k) for k in range(1, 9)} | {(2, 23), (3, 8), (5, 1), (5, 9), (6, 12), (11, 4)}

STOCKS = [{"ticker": s[0], "name": s[1], "sector": s[2], "shares_mln": s[8], "lot": 10 if s[7] < 1000 else 1}
          for s in _STOCKS]


def _t(r, nu):
    """Стандартизованное t-распределение (дисперсия 1): «толстые хвосты»."""
    z = r.gauss(0, 1)
    chi = 2 * r.gammavariate(nu / 2, 1)
    return z / _m.sqrt(chi / nu) / _m.sqrt(nu / (nu - 2))


DATES = []
_d = _D0
while _d <= _D1:
    if _d.weekday() < 5 and (_d.month, _d.day) not in _HOLI:
        DATES.append(_d)
    _d += _td(days=1)
_T = len(DATES)

# ---------- рынок: GJR-GARCH(1,1) с t-шумом, условный кризис весны 2020 ----------
_MU, _OM, _AL, _GA, _BE = 0.00035, 3.0e-6, 0.04, 0.06, 0.90
_rm, _sig = [], []
_s2, _eps = 0.0001, 0.0
for _d in DATES:
    _s2 = _OM + (_AL + _GA * (_eps < 0)) * _eps ** 2 + _BE * _s2 if _rm else _s2
    _z = _t(_R, 6)
    _z = _STRESS.get(_d, _z * (1.15 if _z < 0 else 0.85) + 0.115)  # падения резче роста: отрицательная асимметрия
    _eps = _m.sqrt(_s2) * _z
    _rm.append(_MU + _eps)
    _sig.append(_m.sqrt(_s2))
# ---------- курс и секторы ----------
_fx = [0.00015 - 0.30 * (_rm[i] - _MU) + max(0.003, 0.004 + 0.25 * (_sig[i] - 0.01)) * _t(_R, 5) for i in range(_T)]
_SECT = sorted({s[2] for s in _STOCKS})
_fs = {k: [0.009 * _m.sqrt(_sig[i] / 0.01) * _t(_R, 5) for i in range(_T)] for k in _SECT}

# ---------- новости: шаблоны заголовков ----------
_POS = ["«{n}» увеличивает чистую прибыль на {p} %", "«{n}» повышает прогноз выручки на год",
        "«{n}» объявляет о выкупе акций на {b} млрд руб.", "Рейтинговое агентство повышает рейтинг «{n}»",
        "«{n}» получает крупный контракт на {b} млрд руб.", "Выручка «{n}» растёт на {p} % — выше ожиданий",
        "Совет директоров «{n}» рекомендует рекордные дивиденды", "«{n}» выходит на новый рынок и расширяет производство",
        "Аналитики улучшают оценку «{n}» после сильного отчёта", "«{n}» сокращает долг и снижает расходы"]
_NEG = ["Чистая прибыль «{n}» снижается на {p} %", "«{n}» понижает прогноз выручки на год",
        "«{n}» отменяет выплату дивидендов", "Рейтинговое агентство понижает рейтинг «{n}»",
        "«{n}» теряет крупный контракт", "Выручка «{n}» падает на {p} % — хуже ожиданий",
        "Против «{n}» подан иск на {b} млрд руб.", "«{n}» сообщает об убытке и росте долга",
        "Аналитики ухудшают оценку «{n}» после слабого отчёта", "«{n}» приостанавливает производство из-за аварии"]
_NEU = ["«{n}» проведёт годовое собрание акционеров", "«{n}» опубликует отчётность на следующей неделе",
        "«{n}» назначает нового финансового директора", "«{n}» представит стратегию на дне инвестора",
        "«{n}» меняет аудитора", "«{n}» разместит облигации на {b} млрд руб."]
_MIX = [("Несмотря на снижение выручки, «{n}» увеличивает прибыль на {p} %", 1),
        ("Прибыль «{n}» растёт, но компания понижает прогноз на год", -1),
        ("«{n}» снижает расходы, однако теряет крупный контракт", -1),
        ("Убыток «{n}» сокращается, компания возвращается к росту выручки", 1)]
_LAB = {1: "позитив", -1: "негатив", 0: "нейтрально"}

# ---------- акции ----------
_RS = _random.Random(_SEED + 1)
_rets, _news_eff, NEWS = {}, {}, []
_spread = 0.0
for _s in _STOCKS:
    _tk, _nm, _sec, _b, _g, _sd, _lag = _s[:7]
    _a = 0.00012 if _sec == "энергетика" else (0.0 if _sec == "технологии" else 0.00006)
    _r, _carry, _eff, _h, _e = [], 0.0, {}, 1.0, 0.0
    for _i, _d in enumerate(DATES):
        _lg = (_lag if _d <= _LAG_UNTIL else _LAG_AFTER) if _lag else 0.0
        _h = 0.02 + 0.12 * _e ** 2 + 0.86 * _h                    # собственный GARCH(1,1) шума акции
        _scale = (0.7 + 0.3 * _sig[_i] / 0.01) * _m.sqrt(_h)
        _e = _t(_RS, 5)
        _x = (_a + _b * (_rm[_i] - _MU) + _lg * (_rm[_i - 1] - _MU if _i else 0.0) + _g * _fx[_i] + _fs[_sec][_i]
              + _sd * _scale * _e + _carry)
        _carry = 0.0
        if _tk not in ("KLSU", "TVRA") and _RS.random() < 0.045:                  # новость о компании
            _u = _RS.random()
            _sent = 1 if _u < 0.40 else (-1 if _u < 0.75 else 0)
            _mix = _sent != 0 and _RS.random() < 0.12
            if _mix:
                _tpl, _sent = _RS.choice([x for x in _MIX])
            else:
                _tpl = _RS.choice({1: _POS, -1: _NEG, 0: _NEU}[_sent])
            _jump = _sent * (0.02 + 0.015 * abs(_RS.gauss(0, 1)))
            if _sent and _RS.random() < 0.12:
                _jump *= -0.6                                          # рынок «не согласился» с новостью
            if not _sent:
                _jump = 0.004 * _RS.gauss(0, 1)
            _x += _jump
            _carry = 0.25 * _jump                                      # дрейф после новости на следующий день
            _lab = _sent if _RS.random() > 0.08 else _RS.choice([v for v in (1, -1, 0) if v != _sent])
            _head = _tpl.format(n=_nm, p=_RS.randint(5, 45), b=_RS.randint(2, 60))
            _head = _head[0].upper() + _head[1:]
            NEWS.append({"date": _d.isoformat(), "ticker": _tk, "headline": _head, "label": _LAB[_lab]})
            _eff[_i] = _jump
        _r.append(_x)
    if _tk == "TVRA":                     # коинтеграция с KLSU: спред возвращается к среднему
        _r, _sp, _RC = [], [], _random.Random(_SEED + 5)
        for _i in range(_T):
            _new = 0.96 * _spread + 0.013 * _RC.gauss(0, 1)
            _r.append(_rets["KLSU"][_i] + (_new - _spread))
            _spread = _new
    _rets[_tk], _news_eff[_tk] = _r, _eff

# ---------- инсайдерский след: аномальный объём накануне части позитивных новостей ----------
_RI = _random.Random(_SEED + 9)
_pos_news = [(k, n) for k, n in enumerate(NEWS) if n["label"] == "позитив"]
_INSIDE = {(n["ticker"], n["date"]) for k, n in _RI.sample(_pos_news, 8)}
_POS_IDX = {d.isoformat(): i for i, d in enumerate(DATES)}

# ---------- цены, объёмы, дивиденды, сплит ----------
_RP = _random.Random(_SEED + 2)
PRICES, DIVIDENDS, SPLITS, _BY = [], [], [], {}
for _k, _s in enumerate(_STOCKS):
    _tk, _sd, _p0, _shares, _dy = _s[0], _s[5], _s[7], _s[8], _s[9]
    _p, _rows, _ev = float(_p0), [], []
    _base = 4.0e6 / (_p0 / 100) ** 0.6
    _ins = {_POS_IDX[d] - 1 for t, d in _INSIDE if t == _tk}
    _newsd = set(_news_eff[_tk])
    for _i, _d in enumerate(DATES):
        _r = _rets[_tk][_i]
        _prev = _p
        _div = 0.0
        if _dy and _d.month == 7 and 8 + _k <= _d.day and not any(e[0] == "div" and e[1].year == _d.year for e in _ev):
            _div = round(_dy * _prev * (0.8 + 0.4 * _RP.random()), 2)
            _ev.append(("div", _d, _div))
            DIVIDENDS.append({"ticker": _tk, "ex_date": _d.isoformat(), "dividend": _div})
        _p = _prev * _m.exp(_r) - _div
        if _tk == _SPLIT[0] and _d == _SPLIT[1]:
            _p /= _SPLIT[2]
            _ev.append(("split", _d, _SPLIT[2]))
            SPLITS.append({"ticker": _tk, "date": _d.isoformat(), "ratio": _SPLIT[2]})
        _gap = 0.3 * _r + 0.002 * _RP.gauss(0, 1)
        _o = (_prev - _div) * _m.exp(_gap) / (_SPLIT[2] if (_tk == _SPLIT[0] and _d == _SPLIT[1]) else 1)
        _intra = _sd * (0.6 + 0.4 * _sig[_i] / 0.01) + 0.5 * _sig[_i]
        _hi = max(_o, _p) * _m.exp(abs(_RP.gauss(0, 1)) * _intra * 0.6)
        _lo = min(_o, _p) * _m.exp(-abs(_RP.gauss(0, 1)) * _intra * 0.6)
        _tot = _m.sqrt((_sig[_i]) ** 2 + (_sd * (0.6 + 0.4 * _sig[_i] / 0.01)) ** 2)
        _vol = (_base * (1 + 0.08 * (_d.year - 2019)) * (_sig[_i] / 0.01) ** 0.5
                * _m.exp(0.35 * _RP.gauss(0, 1) + 0.45 * min(abs(_r) / _tot, 5)))
        if _i in _newsd:
            _vol *= 2.5
        if _i in _ins:
            _vol *= 6.0
        if _tk == _SPLIT[0] and _d >= _SPLIT[1]:
            _vol *= _SPLIT[2]
        _nd = 2 if _p < 1000 else 1
        _rows.append([_d.isoformat(), _tk, round(_o, _nd), round(_hi, _nd), round(_lo, _nd), round(_p, _nd),
                      None, int(round(_vol, -1))])
    # скорректированная цена: назад от последней даты, множители дивидендов и сплитов
    _f, _evd = 1.0, {e[1].isoformat(): e for e in _ev}
    for _j in range(len(_rows) - 1, -1, -1):
        _rows[_j][6] = round(_rows[_j][5] * _f, 4)
        e = _evd.get(_rows[_j][0])
        if e is not None and _j:
            _f *= (1 - e[2] / _rows[_j - 1][5]) if e[0] == "div" else 1 / e[2]
    _keys = ("date", "ticker", "open", "high", "low", "close", "adj_close", "volume")
    _BY[_tk] = [dict(zip(_keys, r)) for r in _rows]
for _i in range(_T):
    for _s in _STOCKS:
        PRICES.append(_BY[_s[0]][_i])

# ---------- индекс и курс ----------
_RX = _random.Random(_SEED + 3)
INDEX, FX = [], []
_lv, _usd = 1000.0, 70.0
for _i, _d in enumerate(DATES):
    _o = _lv * _m.exp(0.3 * _rm[_i] + 0.001 * _RX.gauss(0, 1))
    _lv *= _m.exp(_rm[_i])
    _w = 0.6 * _sig[_i]
    INDEX.append({"date": _d.isoformat(), "open": round(_o, 2),
                  "high": round(max(_o, _lv) * _m.exp(abs(_RX.gauss(0, 1)) * _w), 2),
                  "low": round(min(_o, _lv) * _m.exp(-abs(_RX.gauss(0, 1)) * _w), 2), "close": round(_lv, 2)})
    _usd *= _m.exp(_fx[_i])
    FX.append({"date": _d.isoformat(), "usd": round(_usd, 4)})

# ---------- «сырая» выгрузка с дефектами (3 акции) ----------
_RR = _random.Random(_SEED + 7)
RAW_PRICES, RAW_DEFECTS = [], {"пропущенный день": 0, "полный дубль строки": 0, "дата дд.мм.гггг": 0,
                               "цена строкой с запятой": 0, "цена ×100 (ошибка запятой)": 0,
                               "нулевой объём": 0, "зависшая цена (повтор вчерашней)": 0, "пропуск цены": 0}
for _tk in ("BRZB", "KLSU", "PXLS"):
    _prev = None
    for _row in _BY[_tk]:
        _x = {k: _row[k] for k in ("date", "ticker", "open", "high", "low", "close", "volume")}
        _u = _RR.random()
        if _u < 0.012:
            RAW_DEFECTS["пропущенный день"] += 1
            continue
        elif _u < 0.032:
            _x["date"] = _date.fromisoformat(_x["date"]).strftime("%d.%m.%Y"); RAW_DEFECTS["дата дд.мм.гггг"] += 1
        elif _u < 0.047:
            _x["close"] = f"{_x['close']:.2f}".replace(".", ","); RAW_DEFECTS["цена строкой с запятой"] += 1
        elif _u < 0.050:
            _x["close"] = round(_x["close"] * 100, 2); RAW_DEFECTS["цена ×100 (ошибка запятой)"] += 1
        elif _u < 0.056:
            _x["volume"] = 0; RAW_DEFECTS["нулевой объём"] += 1
        elif _u < 0.061 and _prev is not None and isinstance(_prev["close"], float):
            _x["close"] = _prev["close"]
            RAW_DEFECTS["зависшая цена (повтор вчерашней)"] += 1
        elif _u < 0.068:
            _x["close"] = None; RAW_DEFECTS["пропуск цены"] += 1
        RAW_PRICES.append(_x)
        if _RR.random() < 0.01:
            RAW_PRICES.append(dict(_x)); RAW_DEFECTS["полный дубль строки"] += 1
        _prev = _x


def ticks(ticker: str, start: str, end: str, seed: int = _SEED):
    """ЛЕНИВЫЙ генератор сделок за торговые дни [start, end]: кортежи (время «ГГГГ-ММ-ДД ЧЧ:ММ:СС», тикер, цена,
    количество, сторона «B»/«S»). Цены открытия и закрытия дня совпадают с PRICES; около 2–6 тыс. сделок в день."""
    for row in _BY[ticker]:
        if not (start <= row["date"] <= end):
            continue
        r = _random.Random(f"{seed}-{ticker}-{row['date']}")
        n = max(300, min(8000, row["volume"] // 900))
        u = sorted(r.betavariate(0.65, 0.65) for _ in range(n))          # U-образная активность в течение сессии
        lo, lc = _m.log(row["open"]), _m.log(row["close"])
        sig = max(_m.log(row["high"] / row["low"]) / 1.6, 0.002)          # волатильность дня — по диапазону H/L
        w, tp, path = 0.0, 0.0, []
        for x in u:
            w += r.gauss(0, 1) * _m.sqrt(max(x - tp, 1e-9)) * sig
            tp = x
            path.append(w)
        wl = path[-1]
        half = max(0.01, row["close"] * 0.0004)
        for x, wx in zip(u, path):
            side = "B" if r.random() < 0.5 else "S"
            mid = _m.exp(lo + wx - x * wl + x * (lc - lo))
            sec = 36000 + int(x * 31200)                                   # 10:00–18:40
            yield (f"{row['date']} {sec // 3600:02d}:{sec // 60 % 60:02d}:{sec % 60:02d}", ticker,
                   round(mid + (half if side == "B" else -half), 2), int(r.paretovariate(1.6) * 10), side)

print(f"{_TITLE}: акций {len(STOCKS)}, торговых дней {len(DATES)} ({DATES[0]} – {DATES[-1]}), строк котировок {len(PRICES)}, новостей {len(NEWS)}")

In [ ]:
# Инструменты курса «АД и МО на финансовых рынках»: функции, написанные в темах 1–3 (numpy, pandas, scipy).
# В практикумах тем 3–4 эта ячейка идёт сразу после данных кейса.
import numpy as np
import pandas as pd
from scipy import stats


def panel(field="adj_close"):
    """Таблица «дата × тикер» по полю PRICES (по умолчанию — скорректированная цена)."""
    p = pd.DataFrame(PRICES).pivot(index="date", columns="ticker", values=field)
    p.index = pd.to_datetime(p.index)
    return p[[s["ticker"] for s in STOCKS]]


def index_close():
    s = pd.DataFrame(INDEX).set_index("date").close
    s.index = pd.to_datetime(s.index)
    return s


def ols(X, y):
    """МНК со свободным членом: коэффициенты, стандартные ошибки, t-статистики, R²."""
    X = np.column_stack([np.ones(len(y)), np.asarray(X, float)])
    y = np.asarray(y, float)
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    e = y - X @ b
    s2 = e @ e / (len(y) - X.shape[1])
    se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
    return {"coef": b, "se": se, "t": b / se, "r2": 1 - e @ e / ((y - y.mean()) @ (y - y.mean()))}


def fit_logit(X, y, l2=0.0, iters=50):
    """Логистическая регрессия методом Ньютона (IRLS); l2 — штраф на коэффициенты (не на свободный член)."""
    X = np.column_stack([np.ones(len(y)), np.asarray(X, float)])
    y = np.asarray(y, float)
    w = np.zeros(X.shape[1])
    pen = l2 * np.eye(X.shape[1]); pen[0, 0] = 0
    for _ in range(iters):
        p = 1 / (1 + np.exp(-X @ w))
        g = X.T @ (p - y) + pen @ w
        H = X.T @ (X * (p * (1 - p))[:, None]) + pen
        step = np.linalg.solve(H, g)
        w -= step
        if np.abs(step).max() < 1e-8:
            break
    return w


def predict_logit(w, X):
    return 1 / (1 + np.exp(-(w[0] + np.asarray(X, float) @ w[1:])))


def auc(y, score):
    """Площадь под ROC-кривой = вероятность, что случайный «1» получит оценку выше случайного «0»."""
    y = np.asarray(y).astype(bool)
    r = stats.rankdata(score)
    n1, n0 = y.sum(), (~y).sum()
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def roc_points(y, score):
    """Точки ROC-кривой (FPR, TPR) по всем порогам."""
    y = np.asarray(y).astype(bool)
    o = np.argsort(-np.asarray(score), kind="stable")
    tp, fp = np.cumsum(y[o]), np.cumsum(~y[o])
    return np.r_[0, fp / (~y).sum()], np.r_[0, tp / y.sum()]


def gini(y):
    p = np.mean(y) if len(y) else 0.0
    return 2 * p * (1 - p)


def best_split(x, y, min_leaf=20):
    """Лучший порог по одному признаку: минимум взвешенного индекса Джини (векторно, по отсортированным данным)."""
    o = np.argsort(x, kind="stable"); xs, ys = x[o], y[o]
    n = len(y); k = np.arange(1, n)
    c = np.cumsum(ys)[:-1]
    pl, pr = c / k, (ys.sum() - c) / (n - k)
    g = (k * 2 * pl * (1 - pl) + (n - k) * 2 * pr * (1 - pr)) / n
    ok = (k >= min_leaf) & (n - k >= min_leaf) & (xs[1:] > xs[:-1])
    if not ok.any():
        return None, np.inf
    j = np.argmin(np.where(ok, g, np.inf))
    return (xs[j] + xs[j + 1]) / 2, g[j]


def fit_tree(X, y, depth=3, min_leaf=20, features=None, rng=None):
    """Дерево классификации CART. Узел — словарь: лист {"p": доля 1} или {"f", "thr", "left", "right"}."""
    X, y = np.asarray(X, float), np.asarray(y, float)
    node = {"p": y.mean() if len(y) else 0.5, "n": len(y)}
    if depth == 0 or len(y) < 2 * min_leaf or y.min() == y.max():
        return node
    cols = range(X.shape[1]) if features is None else rng.choice(X.shape[1], features, replace=False)
    best = (None, None, gini(y))
    for f in cols:
        thr, g = best_split(X[:, f], y, min_leaf)
        if thr is not None and g < best[2] - 1e-12:
            best = (f, thr, g)
    if best[0] is None:
        return node
    f, thr = best[0], best[1]
    m = X[:, f] <= thr
    node.update(f=int(f), thr=float(thr),
                left=fit_tree(X[m], y[m], depth - 1, min_leaf, features, rng),
                right=fit_tree(X[~m], y[~m], depth - 1, min_leaf, features, rng))
    return node


def predict_tree(node, X):
    X = np.asarray(X, float)
    out = np.empty(len(X))
    for i, x in enumerate(X):
        nd = node
        while "f" in nd:
            nd = nd["left"] if x[nd["f"]] <= nd["thr"] else nd["right"]
        out[i] = nd["p"]
    return out


def fit_forest(X, y, n_trees=100, depth=4, min_leaf=30, features=None, seed=0):
    """Случайный лес: деревья на бутстреп-выборках со случайным подмножеством признаков в каждом узле."""
    X, y = np.asarray(X, float), np.asarray(y, float)
    rng = np.random.default_rng(seed)
    k = features or max(1, int(np.sqrt(X.shape[1])))
    trees = []
    for _ in range(n_trees):
        i = rng.integers(0, len(y), len(y))
        trees.append(fit_tree(X[i], y[i], depth, min_leaf, k, rng))
    return trees


def predict_forest(trees, X):
    return np.mean([predict_tree(t, X) for t in trees], axis=0)


def make_features(ticker):
    """Признаки на закрытие дня t и цель — рост акции в день t+1 (y = 1). Только прошлое и настоящее дня t."""
    px, vol = panel()[ticker], panel("volume")[ticker]
    idx, usd = index_close(), pd.DataFrame(FX).set_index("date").usd
    usd.index = pd.to_datetime(usd.index)
    r, ri = np.log(px).diff(), np.log(idx).diff()
    up, dn = r.clip(lower=0).rolling(14).mean(), (-r).clip(lower=0).rolling(14).mean()
    news = pd.DataFrame(NEWS)
    news = news[news.ticker == ticker].assign(s=lambda d: d.label.map({"позитив": 1, "негатив": -1, "нейтрально": 0}))
    sent = news.groupby(pd.to_datetime(news.date)).s.sum().reindex(px.index, fill_value=0)
    f = pd.DataFrame({
        "idx_r0": ri, "idx_r5": ri.rolling(5).sum(), "own_r0": r, "own_r5": r.rolling(5).sum(),
        "own_r20": r.rolling(20).sum(), "vol20": r.rolling(20).std(), "idx_vol20": ri.rolling(20).std(),
        "vol_z": (np.log(vol) - np.log(vol).rolling(20).mean()) / np.log(vol).rolling(20).std(),
        "ma_gap": px / px.rolling(20).mean() - 1, "rsi14": 100 * up / (up + dn),
        "fx_r0": np.log(usd).diff(), "news0": sent})
    f["r_next"] = r.shift(-1)
    f["y"] = (f.r_next > 0).astype(int)
    return f.dropna()


FEATURES = ["idx_r0", "idx_r5", "own_r0", "own_r5", "own_r20", "vol20", "idx_vol20", "vol_z", "ma_gap", "rsi14",
            "fx_r0", "news0"]


def standardize(train, test):
    """Масштабирование по среднему и ст. отклонению ОБУЧАЮЩЕЙ части (никаких сведений из теста)."""
    m, s = train.mean(), train.std().replace(0, 1)
    return (train - m) / s, (test - m) / s


def walk_forward(df, fit, predict, cols, start="2022-01-01", step=63, min_train=500):
    """Расширяющееся окно: обучение на всём до блока, прогноз на следующие step дней, сдвиг на step."""
    idx = df.index
    pos = int(np.searchsorted(idx, pd.Timestamp(start)))
    out = []
    while pos < len(df):
        tr, te = df.iloc[:pos], df.iloc[pos:pos + step]
        if len(tr) >= min_train:
            a, b = standardize(tr[cols], te[cols])
            model = fit(a.to_numpy(), tr.y.to_numpy())
            out.append(pd.Series(predict(model, b.to_numpy()), index=te.index))
        pos += step
    return pd.concat(out)


def sharpe(r, periods=252):
    r = pd.Series(r).dropna()
    return float(r.mean() / r.std() * np.sqrt(periods)) if r.std() > 0 else 0.0


def max_drawdown(r):
    """Максимальная просадка по ряду лог-доходностей (доля от пика, отрицательное число)."""
    w = np.exp(pd.Series(r).fillna(0).cumsum())
    return float((w / w.cummax() - 1).min())

print("Инструменты курса загружены:", ", ".join(sorted(k for k in dir() if k in {"panel", "index_close", "ols", "fit_logit", "predict_logit", "auc", "roc_points", "fit_tree", "predict_tree", "fit_forest", "predict_forest", "make_features", "walk_forward", "sharpe", "max_drawdown"})))

## 1. Walk-forward 📘

Каждый прогноз сделан моделью, обученной только на прошлом.

In [ ]:
# Walk-forward: переобучение каждые 63 дня на всей предыдущей истории (функция из «Инструментов курса»)
f = make_features("KLSU")
fit = lambda X, y: fit_logit(X, y, l2=1.0)
p = walk_forward(f, fit, predict_logit, ["idx_r0"], start="2021-01-01", step=63)
d = f.loc[p.index].assign(p=p)
x = d.loc[:"2024-06"]                                   # период «живого» сигнала
print(f"прогнозов {len(p)}; AUC 2021 — 06.2024: {auc(x.y, x.p):.3f}; "
      f"после: {auc(d.y.loc['2024-07':], d.p.loc['2024-07':]):.3f}")

## 2. Случайная кросс-валидация обманывает 📘

Перекрывающиеся метки + перемешивание = утечка.

In [ ]:
# Перекрывающиеся 5-дневные метки: случайная кросс-валидация против деления по времени
r = np.log(panel()["KLSU"]).diff()
g = f.assign(y5=(r.rolling(5).sum().shift(-5) > 0).astype(int)).dropna().loc[:"2024-06"]
cols = [c for c in FEATURES if g[c].std() > 0]
X = ((g[cols] - g[cols].mean()) / g[cols].std()).to_numpy(); Y = g.y5.to_numpy()
idx = np.random.default_rng(0).permutation(len(Y))
rand = [(np.setdiff1d(idx, k), k) for k in np.array_split(idx, 5)]
blocks = np.array_split(np.arange(len(Y)), 5)
wf = [(np.arange(b[0]), b) for b in blocks[1:]]
for name, splits in [("случайно", rand), ("по времени", wf)]:
    a = [auc(Y[te], predict_forest(fit_forest(X[tr], Y[tr], 60, 8, 5, seed=1), X[te])) for tr, te in splits]
    print(f"лес, {name}: AUC = {np.mean(a):.3f}")

## 3. Утечки в признаках 📘

Будущее в признаке «улучшает» и обучение, и тест.

In [ ]:
# Утечки: рынок завтра и центрированная средняя
px = panel()["KLSU"]
ri = np.log(index_close()).diff()
h = f.assign(idx_r1=ri.shift(-1), ma_c=px / px.rolling(21, center=True).mean() - 1).dropna()
tr, te = h.loc[:"2022"], h.loc["2023":"2024-06"]
for cs in [["idx_r0"], ["idx_r0", "idx_r1"], ["idx_r0", "ma_c"]]:
    m, s = tr[cs].mean(), tr[cs].std()
    w_ = fit_logit((tr[cs] - m) / s, tr.y, l2=1.0)
    print(f"{' + '.join(cs):<16} AUC тест {auc(te.y, predict_logit(w_, (te[cs] - m) / s)):.3f}")

## 4. ROC, AUC и интервал 📘

Блочный бутстреп сохраняет зависимость соседних дней.

In [ ]:
# ROC-кривая, AUC и блочный бутстреп
fpr, tpr = roc_points(x.y, x.p)
rng = np.random.default_rng(1)
bl = np.array_split(np.arange(len(x)), len(x) // 21)
boot = [auc(x.y.to_numpy()[q], x.p.to_numpy()[q])
        for q in (np.concatenate([bl[i] for i in rng.integers(0, len(bl), len(bl))]) for _ in range(500))]
print(f"AUC {auc(x.y, x.p):.3f}, 95 % интервал [{np.quantile(boot, .025):.3f}; {np.quantile(boot, .975):.3f}]")
plt.figure(figsize=(4.5, 4)); plt.plot(fpr, tpr, color=BLUE); plt.plot([0, 1], [0, 1], "--", color=GREY)
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC: KLSU, walk-forward"); plt.show()

## 5. Калибровка 📘

Log-loss и Brier против наивного прогноза; прогноз и факт по квинтилям.

In [ ]:
# Вероятности: log-loss, Brier, калибровка по квинтилям прогноза
base = np.full(len(x), x.y.mean())
ll = lambda q: -np.mean(x.y * np.log(q) + (1 - x.y) * np.log(1 - q))
print(f"log-loss {ll(x.p):.4f} (наивный {ll(base):.4f}); Brier {np.mean((x.p - x.y) ** 2):.4f} "
      f"(наивный {np.mean((base - x.y) ** 2):.4f})")
x.groupby(pd.qcut(x.p, 5), observed=True).agg(прогноз=("p", "mean"), факт=("y", "mean")).round(3)

## 6. Гиперпараметры 📘

Подбор на валидации; тест — один раз.

In [ ]:
# Подбор λ на валидации по времени, итог — на нетронутом тесте
tr, va, te = f.loc[:"2021"], f.loc["2022"], f.loc["2023":"2024-06"]
cs = [c for c in FEATURES if f[c].std() > 0]
m, s = tr[cs].mean(), tr[cs].std()
grid = {lam: auc(va.y, predict_logit(fit_logit((tr[cs] - m) / s, tr.y, l2=lam), (va[cs] - m) / s))
        for lam in [0.1, 1, 10, 100, 1000]}
print({k: round(v, 3) for k, v in grid.items()})

## 7. Перестановочная важность 📘

Падение AUC при перемешивании признака.

In [ ]:
# Перестановочная важность (лес на 2019–2022, тест 2023 — 06.2024)
trva = f.loc[:"2022"]
m, s = trva[cs].mean(), trva[cs].std()
A, B = ((trva[cs] - m) / s).to_numpy(), ((te[cs] - m) / s).to_numpy()
forest = fit_forest(A, trva.y.to_numpy(), n_trees=100, depth=4, min_leaf=30, seed=3)
base_auc = auc(te.y, predict_forest(forest, B))
rng = np.random.default_rng(7)
imp = {}
for j, c in enumerate(cs):
    Bp = B.copy(); Bp[:, j] = rng.permutation(Bp[:, j])
    imp[c] = base_auc - auc(te.y, predict_forest(forest, Bp))
pd.Series(imp).sort_values(ascending=False).round(4)

## 8. Бэктест с издержками 📘

Оборот и издержки решают судьбу стратегии.

In [ ]:
# Бэктест long/short с издержками
pos = np.sign(d.p - 0.5)
turn = pos.diff().abs().fillna(1.0)
gross = pos * d.r_next
for cost in [0.0, 0.001, 0.002]:
    net = gross - turn * cost
    print(f"издержки {cost:.1%}: Шарп 2021 — 06.2024 = {sharpe(net.loc[:'2024-06']):.2f}, "
          f"после = {sharpe(net.loc['2024-07':]):.2f}")
eq = np.exp((gross - turn * 0.001).cumsum())
eq.plot(logy=True, figsize=(9, 3), title="капитал стратегии, издержки 0,1 %"); plt.axvline("2024-07-01", color=GREY); plt.show()

## 9. Мониторинг 📘

PSI признака не видит дрейфа концепции; скользящий AUC — видит, но с задержкой.

In [ ]:
# Мониторинг: PSI признака и скользящий AUC
def psi(ref, cur, bins=10):
    q = np.quantile(ref, np.linspace(0, 1, bins + 1)); q[0], q[-1] = -np.inf, np.inf
    a = np.clip(np.histogram(ref, q)[0] / len(ref), 1e-6, None)
    b = np.clip(np.histogram(cur, q)[0] / len(cur), 1e-6, None)
    return float(np.sum((b - a) * np.log(b / a)))

print({y_: round(psi(f.idx_r0.loc[:"2022"], f.idx_r0.loc[y_]), 3) for y_ in ["2023", "2024", "2025", "2026"]})
roll = pd.Series([auc(d.y.iloc[i - 126:i], d.p.iloc[i - 126:i]) for i in range(126, len(d) + 1)], index=d.index[125:])
roll.plot(figsize=(9, 3), title="скользящий AUC, 126 дней"); plt.axhline(0.52, color=ORANGE, ls="--"); plt.show()

---

# Задания ✏️

Пишите код вместо `...` / `# ваш код`, затем запускайте **ячейку проверки** под заданием.

## Задание 1. Пороговые метрики

Accuracy, precision, recall, F1, FPR.

In [ ]:
# Задание 1. Функция metrics(y, p, thr) -> dict с ключами "accuracy", "precision", "recall", "f1", "fpr"
# (значения округлите до 4 знаков). Примените к прогнозам x (KLSU, 2021 — 06.2024) при порогах 0,5 и 0,55:
# m50, m55. Как порог меняет precision и recall?
def metrics(y, p, thr=0.5):
    ...

In [ ]:
assert m50 == {'accuracy': 0.5587, 'precision': 0.5583, 'recall': 0.5679, 'f1': 0.563, 'fpr': 0.4505}
assert m55 == {'accuracy': 0.5352, 'precision': 0.679, 'recall': 0.1358, 'f1': 0.2263, 'fpr': 0.0644}
print("Задание 1: верно")

## Задание 2. ROC своими руками

Накопленные доли и трапеции.

In [ ]:
# Задание 2. ROC-кривая своими руками для прогнозов x: отсортируйте дни по убыванию p, накопленные доли
# TPR и FPR (начиная с точки (0, 0)), площадь — трапециями: my_auc. Сравните с auc(x.y, x.p).
# ваш код

In [ ]:
assert abs(my_auc - auc(x.y, x.p)) < 1e-9
assert tpr_[0] == 0 and fpr_[-1] == 1
print("Задание 2: верно")

## Задание 3. Скользящее окно

500 последних дней вместо всей истории.

In [ ]:
# Задание 3. Walk-forward со СКОЛЬЗЯЩИМ окном: на каждом шаге (63 дня, начиная с 2021-01-01) обучайтесь только
# на последних 500 днях до блока (признак idx_r0, масштаб по окну, fit_logit l2=1). Прогнозы — Series ps.
# slide = {"до 07.2024": AUC на ps до 30.06.2024, "после": AUC с 01.07.2024} (округлить до 4 знаков).
# ваш код

In [ ]:
assert slide == {'до 07.2024': 0.5872, 'после': 0.5249}
print("Задание 3: верно")

## Задание 4. Найдите утечку

Подозрительно хороший признак.

In [ ]:
# Задание 4. Найдите утечку. В таблице sus четыре признака; один из них заглядывает в будущее. Для каждого признака
# посчитайте AUC на тесте 2023 — 06.2024 «как есть» (score = значение признака): Series auc_sus. leak_name — имя
# подозрительного признака. Затем исправьте его: honest — тот же признак, но окно только по прошлому (дни t−2..t),
# и посчитайте auc_honest.
r1 = np.log(panel()["KLSU"]).diff()
sus = pd.DataFrame({"idx_r0": f.idx_r0, "own_r5": f.own_r5, "vol_z": f.vol_z,
                    "mom3": r1.rolling(3, center=True).sum().loc[f.index]}).dropna()
# ваш код

In [ ]:
assert leak_name == "mom3" and auc_sus["mom3"] > 0.6
assert auc_honest < 0.56
print("Задание 4: верно")

## Задание 5. Калибровка Платта

Логистическая регрессия на логит прогноза.

In [ ]:
# Задание 5. Калибровка Платта: на прогнозах 2021–2022 (d) обучите логистическую регрессию y на logit(p)
# (fit_logit, без штрафа), примените к прогнозам 2023 — 06.2024: p_cal. brier = {"до": Brier исходных прогнозов
# 2023 — 06.2024, "после": Brier p_cal} (округлить до 5 знаков). Улучшилась ли калибровка?
# ваш код

In [ ]:
assert brier == {'до': 0.23837, 'после': 0.23974}
print("Задание 5: верно")

## Задание 6. Порог уверенности

Выбор δ на валидации, проверка на тесте.

In [ ]:
# Задание 6. Торговать только уверенными сигналами: позиция +1, если p − 0,5 > δ; −1, если 0,5 − p > δ; иначе 0.
# Функция bt(dd, delta, cost=0.001) -> (Шарп с издержками, оборот). Выберите δ из deltas по Шарпу на 2021–2022
# (sh_val — dict {δ: Шарп, округлить до 3 знаков}, best_delta), затем оцените на 2023 — 06.2024: sh_test.
deltas = [0.0, 0.005, 0.01, 0.02, 0.03]
def bt(dd, delta, cost=0.001):
    ...

In [ ]:
assert sh_val == {0.0: 0.067, 0.005: -0.172, 0.01: -0.118, 0.02: -0.443, 0.03: 0.317}
assert best_delta == 0.03 and sh_test == 3.024
print("Задание 6: верно")

## Задание 7. Важность для логита

Перестановки для линейной модели.

In [ ]:
# Задание 7. Перестановочная важность для ЛОГИСТИЧЕСКОЙ регрессии (l2 = 10, обучение trva, признаки cs, тест te):
# по 10 перестановок на признак (rng = np.random.default_rng(3)); imp_logit — Series среднего падения AUC,
# отсортированная по убыванию. top — самый важный признак. Совпадает ли с лесом из разбора?
# ваш код

In [ ]:
assert top == "idx_r0" and imp_logit.iloc[0] > 0.03
assert len(imp_logit) == len(cs)
print("Задание 7: верно")

## Задание 8 ★. Мини-исследование: правило остановки

Когда выключать модель.

In [ ]:
# Задание 8 ★ (мини-исследование). Правило остановки модели. Стратегия из разбора (long/short, издержки 0,1 %).
# Правило: с 01.01.2022 каждый день проверяем скользящий AUC за 126 дней (roll из разбора); если он ниже порога thr,
# с ЗАВТРАШНЕГО дня стратегия выключается навсегда (позиция 0). Для thr из [0.50, 0.52, 0.54]: дата остановки,
# доход стратегии с остановкой и без неё за 07.2024 — 09.2026 и за весь период (сумма лог-доходностей, %).
# Таблица stop (индекс — порог). Вывод 5–7 предложений: какой порог разумен, чем плох слишком чувствительный.
# ваш код

In [ ]:
assert stop.shape == (3, 5)
assert stop["без остановки, весь период, %"].nunique() == 1
print("Задание 8: таблица построена; вывод проверит преподаватель")

---

## Что вы теперь умеете

| Задача | Инструмент |
|---|---|
| проверка по времени | `walk_forward`, скользящее окно, очистка и эмбарго |
| утечки | центрированные окна, `shift(-1)`, масштаб по всей выборке |
| метрики | матрица ошибок, `roc_points`, `auc`, блочный бутстреп, Brier |
| гиперпараметры и объяснение | валидация по времени, перестановочная важность |
| бэктест | позиция, оборот, издержки, Шарп, просадка, безубыточные издержки |
| мониторинг | PSI, скользящий AUC, правило остановки |

**Дальше — тема 4 «Продвинутые методы и практические кейсы».** Нейросети и смешивание
моделей, анализ новостей, тиковые данные, байесовские методы и интеграция с торговыми системами.